# NEM SA1 — EDA & Signal Identification

**Goal for today**: Load the shared prepared dataset and identify which signals have the strongest relationship with `RRP`. No modelling yet.

## Data source
This notebook does **not** download raw AEMO data directly.

It expects the shared dataset to already be built by:

- `NEM_Extractor_SA1_V1.ipynb`

Expected input file:

- `../../data/sa1_merged_eda.csv`

### Tables used
| Table | Why |
|---|---|
| `DISPATCHPRICE` | `RRP` — target variable + FCAS prices |
| `DISPATCHREGIONSUM` | Demand, supply, curtailment signals |
| `DISPATCHINTERCONNECTORRES` | SA interconnector flow (Heywood + Murraylink) |


> **`INTERVENTION` note**: All three tables are filtered to `INTERVENTION=0` (pricing run).
> Intervention=1 records are physical dispatch targets, not prices — keeping both doubles rows and corrupts the merge.

## 1. Setup

This notebook loads the prepared shared dataset created by `Data_Extraction_SA1.ipynb`.

It does **not** download raw AEMO data directly.

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels --quiet

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 4)

os.makedirs('../../data', exist_ok=True)
DATA_DIR = '../../data'

## 2. Load Shared Dataset

Load the prepared SA1 dataset built by the standalone extraction notebook.

In [ ]:
DATA_DIR = '../../data'
df = pd.read_csv(os.path.join(DATA_DIR, 'sa1_merged_eda.csv'), parse_dates=['SETTLEMENTDATE'])

# Make sure the data loaded correctly
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range:    {df.SETTLEMENTDATE.min()} to {df.SETTLEMENTDATE.max()}')
print(f'\nColumn list:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')

## 3. Data Quality Check

In [ ]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Missing values:')
print(missing)

# Expected intervals: 2 years x 365.25 days x 48 half-hour intervals/day
expected = int(2 * 365.25 * 48)
print(f'\nExpected intervals (~2 years, 30-minute data): {expected:,}')
print(f'Actual intervals: {len(df):,}')
print(f'Missing intervals: {expected - len(df):,}')

## 4. Inspect the Target Variable: RRP

Before touching features — understand what you're trying to predict.

In [ ]:
print('=== RRP Summary Statistics ===')
print(df['RRP'].describe())
print(f'\nNegative price intervals: {(df.RRP < 0).sum():,} ({(df.RRP < 0).mean():.1%})')
print(f'Intervals below -$100/MWh: {(df.RRP < -100).sum():,} ({(df.RRP < -100).mean():.2%})')
print(f'Intervals above $1000/MWh: {(df.RRP > 1000).sum():,} ({(df.RRP > 1000).mean():.2%})')
print(f'Intervals above $5000/MWh: {(df.RRP > 5000).sum():,} ({(df.RRP > 5000).mean():.2%})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Full distribution (log scale to show the tails)
axes[0].hist(df['RRP'], bins=200, color='steelblue', edgecolor='none')
axes[0].set_yscale('log')
axes[0].set_xlabel('RRP ($/MWh)')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('RRP distribution (full range)')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1, label='$0')
axes[0].legend()

# Zoomed: -200 to 500 to see the bulk + negative tail
clipped = df['RRP'].clip(-300, 500)
axes[1].hist(clipped, bins=150, color='steelblue', edgecolor='none')
axes[1].set_xlabel('RRP ($/MWh)')
axes[1].set_ylabel('Count')
axes[1].set_title('RRP distribution (clipped -$300 to $500)')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1, label='$0')
axes[1].legend()

# Time series — 4 week sample to show volatility clustering
sample = df.set_index('SETTLEMENTDATE')['RRP'].loc['2024-01':'2024-02']
axes[2].plot(sample.index, sample.values, linewidth=0.4, color='steelblue')
axes[2].axhline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('RRP ($/MWh)')
axes[2].set_title('RRP time series — Jan/Feb 2024')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.tight_layout()
plt.savefig('../../data/rrp_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: ../../data/rrp_distribution.png')

In [ ]:
# Full 2-year time series — shows volatility clustering clearly
fig, ax = plt.subplots(figsize=(16, 4))
ts = df.set_index('SETTLEMENTDATE')['RRP']
ax.plot(ts.index, ts.values, linewidth=0.3, color='steelblue', alpha=0.8)
ax.axhline(0, color='red', linestyle='--', linewidth=0.8, label='$0')
ax.set_xlabel('Date')
ax.set_ylabel('RRP ($/MWh)')
ax.set_title('SA1 RRP — 2023 to 2024 (30-min intervals)')
ax.legend()
plt.tight_layout()
plt.savefig('../../data/rrp_timeseries.png', bbox_inches='tight')
plt.show()

## 6. Autocorrelation of RRP

How much memory does the price series have? 
- ACF tells you the direct + indirect correlation at each lag
- PACF tells you the direct correlation once shorter lags are controlled for

These plots directly inform GARCH(p,q) parameter selection.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf

rrp = df['RRP'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(rrp, lags=60, ax=axes[0], title='ACF — RRP (60 lags = 30 hours)')
plot_pacf(rrp, lags=60, ax=axes[1], title='PACF — RRP (60 lags = 30 hours)')
axes[0].set_xlabel('Lag (30-min intervals)')
axes[1].set_xlabel('Lag (30-min intervals)')
plt.tight_layout()
plt.savefig('../../data/rrp_acf_pacf.png', bbox_inches='tight')
plt.show()

# Print ACF values at meaningful lags
acf_vals = acf(rrp, nlags=288)
lags_of_interest = {1: '30 min', 2: '1 hour', 6: '3 hours', 12: '6 hours', 24: '12 hours', 48: '24 hours'}
print('ACF at key lags:')
for lag, label in lags_of_interest.items():
    print(f'  Lag {lag:3d} ({label:>8}): {acf_vals[lag]:.4f}')

In [ ]:
# ACF of squared returns — this is the key GARCH diagnostic
# If squared RRP shows strong autocorrelation, volatility is clustering -> GARCH is motivated
rrp_sq = rrp ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(rrp_sq, lags=60, ax=axes[0], title='ACF — RRP² (volatility clustering)')
plot_pacf(rrp_sq, lags=60, ax=axes[1], title='PACF — RRP²')
axes[0].set_xlabel('Lag (30-min intervals)')
axes[1].set_xlabel('Lag (30-min intervals)')
plt.tight_layout()
plt.savefig('../../data/rrp_squared_acf.png', bbox_inches='tight')
plt.show()

print('If ACF of RRP² is significant at many lags, volatility is clustering.')
print('This is the primary empirical justification for GARCH over simpler models.')

## 7. Pearson & Spearman Correlation Against RRP

- **Pearson**: linear relationships
- **Spearman**: monotonic relationships (better for extreme values / nonlinear)

Where the two disagree strongly, the relationship is nonlinear.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['SETTLEMENTDATE', 'RRP']]

pearson  = df[feature_cols + ['RRP']].corr(method='pearson')['RRP'].drop('RRP')
spearman = df[feature_cols + ['RRP']].corr(method='spearman')['RRP'].drop('RRP')

corr_df = pd.DataFrame({
    'Pearson':  pearson,
    'Spearman': spearman,
    'Divergence': (pearson - spearman).abs()  # large = nonlinear relationship
}).sort_values('Spearman', key=abs, ascending=False)

print('=== Correlation with RRP (sorted by |Spearman|) ===')
print(corr_df.round(4).to_string())

In [ ]:
# Visual: ranked bar chart of Spearman correlations
sorted_corr = corr_df['Spearman'].sort_values()

fig, ax = plt.subplots(figsize=(10, max(6, len(sorted_corr) * 0.35)))
colors = ['#d73027' if v < 0 else '#4575b4' for v in sorted_corr.values]
ax.barh(sorted_corr.index, sorted_corr.values, color=colors, edgecolor='none', height=0.7)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Spearman correlation with RRP')
ax.set_title('Feature correlations with SA1 RRP (Spearman)')
ax.grid(axis='x', linewidth=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig('../../data/correlation_ranked.png', bbox_inches='tight')
plt.show()

In [ ]:
# Highlight where Pearson and Spearman diverge most
print('=== Largest Pearson vs Spearman divergence (nonlinear relationships) ===')
print(corr_df.sort_values('Divergence', ascending=False).head(10).round(4).to_string())

## 8. Feature Correlation Heatmap

Check for multicollinearity between features. Highly correlated feature pairs add noise without adding information — one should be dropped before modelling.

In [ ]:
# Use only the top features by |Spearman| to keep the heatmap readable
top_features = corr_df['Spearman'].abs().sort_values(ascending=False).head(15).index.tolist()
top_features = ['RRP'] + top_features

corr_matrix = df[top_features].corr(method='spearman')

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    ax=ax, square=True,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Spearman correlation matrix — top 15 features + RRP')
plt.tight_layout()
plt.savefig('../../data/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 9. Scatter Plots: Top Features vs RRP

Visual check of the relationship shape for the strongest signals. Look for:
- Linearity vs nonlinearity
- Heteroskedasticity (variance changes across the range)
- Whether extreme RRP values cluster at specific feature values

In [ ]:
top5 = corr_df['Spearman'].abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# Clip RRP for visibility — extreme spikes compress everything else
rrp_clipped = df['RRP'].clip(-500, 500)

for i, feat in enumerate(top5):
    ax = axes[i]
    sc = ax.scatter(
        df[feat], rrp_clipped,
        alpha=0.05, s=1, c=rrp_clipped, cmap='RdBu_r',
        vmin=-300, vmax=300
    )
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('RRP clipped ±$500', fontsize=9)
    r_sp = corr_df.loc[feat, 'Spearman']
    r_pe = corr_df.loc[feat, 'Pearson']
    ax.set_title(f'{feat}\nSpearman={r_sp:.3f}  Pearson={r_pe:.3f}', fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Top features vs RRP (clipped ±$500)', y=1.01)
plt.tight_layout()
plt.savefig('../../data/scatter_top_features.png', bbox_inches='tight')
plt.show()

## 10. Lag Correlation Analysis

Does the current value of a feature predict RRP better at a lag?
Also: how far back does RRP itself predict future RRP?

This informs LSTM window size and GARCH lag terms.

In [ ]:
top5 = corr_df['Spearman'].abs().sort_values(ascending=False).head(5).index.tolist()
lag_features = ['RRP'] + [f for f in top5 if f != 'RRP']
max_lags = 72

# Pre-rank once — use only the unique features
ranked = df[lag_features].rank()

lag_corrs = {}
for feat in lag_features:
    print(f'Processing {feat}...')
    lag_corrs[feat] = [
        ranked['RRP'].corr(ranked[feat].shift(k))
        for k in range(0, max_lags + 1)
    ]
    print(f'  done ({max_lags + 1} lags)')

lag_df = pd.DataFrame(lag_corrs, index=range(0, max_lags + 1))
print('Lag correlations computed.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for feat in lag_features:
    ax.plot(lag_df.index, lag_df[feat], label=feat, linewidth=1.2)

ax.axhline(0, color='black', linewidth=0.6)
ax.set_xlabel('Lag k (30-min intervals) — feature at t-k vs RRP at t')
ax.set_ylabel('Spearman correlation')
ax.set_title('Lag correlation with RRP (0 to 72 lags = 0 to 36 hours)')
ax.legend(fontsize=8, loc='upper right')

# Add time labels on x-axis
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
tick_lags = [0, 6, 12, 24, 36, 48, 60, 72]
tick_labels = ['0', '3h', '6h', '12h', '18h', '24h', '30h', '36h']
ax2.set_xticks(tick_lags)
ax2.set_xticklabels(tick_labels, fontsize=8)
ax2.set_xlabel('Lag time')

plt.tight_layout()
plt.savefig('../../data/lag_correlation.png', bbox_inches='tight')
plt.show()

## 11. Summary: Ranked Signal Strength

In [ ]:
print('=' * 65)
print('SIGNAL STRENGTH SUMMARY — Spearman correlation with RRP')
print('=' * 65)
print(f'{"Feature":<40} {"Spearman":>10} {"Pearson":>10} {"Nonlinear?":>12}')
print('-' * 65)
for feat, row in corr_df.iterrows():
    nonlinear = '  YES' if row['Divergence'] > 0.1 else ''
    print(f'{feat:<40} {row["Spearman"]:>10.4f} {row["Pearson"]:>10.4f} {nonlinear:>12}')
print('=' * 65)
print('\nNonlinear = Pearson vs Spearman divergence > 0.1')
print('Negative correlation = feature increasing -> price decreasing (e.g. more wind -> lower price)')

In [ ]:
corr_df.to_csv('../../data/sa1_correlations.csv')

print('Saved:')
print('  ../../data/sa1_correlations.csv      <- correlation table')
print('  ../../data/rrp_distribution.png      <- target distribution plots')
print('  ../../data/rrp_timeseries.png        <- full RRP time series')
print('  ../../data/rrp_acf_pacf.png          <- ACF and PACF plots')
print('  ../../data/rrp_squared_acf.png       <- squared RRP autocorrelation')
print('  ../../data/correlation_ranked.png    <- ranked feature correlations')
print('  ../../data/correlation_heatmap.png   <- top feature heatmap')
print('  ../../data/scatter_top_features.png  <- top feature scatter plots')
print('  ../../data/lag_correlation.png       <- lag correlation chart')